<a href="https://colab.research.google.com/github/nguyenminhvuinfo/medgemma/blob/main/med_gemma.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q -U transformers huggingface_hub accelerate bitsandbytes

In [ ]:
from google.colab import drive
import pandas as pd

drive.mount('/content/drive')

file_path = '/content/drive/MyDrive/thuc-tap/datasets/discharge_sample_100_patients_gt5.csv.gz'
df_sample = pd.read_csv(file_path, compression='gzip')


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from google.colab import userdata
from huggingface_hub import login

hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)

model_id = "google/medgemma-1.5-4b-it"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/115 [00:00<?, ?B/s]

In [ ]:
import torch

raw_text = df_sample['text'].iloc[0]
first_charttime = df_sample['charttime'].iloc[0]

messages = [
    {"role": "user", "content": f"""You are an expert doctor reviewing a patient's case.
Read the clinical text below and provide a BRIEF, high-level narrative summary of the patient's presentation, hospital course, and outcome.
Write it as a single paragraph of no more than 4-5 sentences.

CLINICAL TEXT:
{raw_text}"""}
]

prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=800,
        temperature=0.05,
        do_sample=False,
        repetition_penalty=1.15
    )

input_length = inputs['input_ids'].shape[1]
result = tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True)
summary_text = result.strip()

print(f"CHARTTIME: {first_charttime}")
print("-" * 50)
print(summary_text)

Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


CHARTTIME: 2183-08-21 00:00:00
--------------------------------------------------
***
**Patient Presentation:** This is a 60-year-old male with a complex medical history including congestive heart failure (CHF), chronic obstructive pulmonary disease (COPD), hypertension (HTN), a history of pulmonary emboli (PE) requiring anticoagulation (though currently off), past heroin use treated with methadone, hepatitis infections (resolved), multiple psychiatric diagnoses (including antisocial personality disorder, depression, anxiety, PTSD), microcytic anemia, vitamin B12 deficiency, GERD, and benign prostatic hyperplasia (BPH). He presented to the hospital with worsening bilateral leg pain, edema, and chest wall pain over three days, accompanied by shortness of breath (SOB) at rest. His vital signs upon arrival were notable for tachycardia (HR 66) and borderline hypertension (BP 163/84). Initial workup revealed unremarkable chest X-ray (CXR), slightly elevated BNP, and an unchanged ventilation